In [1]:
# === Librerías básicas ===
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, accuracy_score
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
import joblib


html = f"""
<link href="https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700&family=Poppins:wght@300;400;600&display=swap" rel="stylesheet">
<style>
:root {{
--neon1: #00ffff;
--neon2: #ff00ff;
--ink: rgba(255,255,255,.05);
}}
body {{
margin:0;
background: radial-gradient(circle at top, #15002b 0%, #050010 55%, #000 100%);
font-family: 'Poppins', sans-serif;
color: #e7f5ff;
overflow-x: hidden;
}}
.bg {{
position:fixed; inset:0;
background:
radial-gradient(800px 650px at 10% 10%, rgba(0,255,255,.18), transparent 60%),
radial-gradient(600px 500px at 90% 25%, rgba(255,0,255,.2), transparent 60%),
linear-gradient(160deg, rgba(0,0,0,0) 0%, rgba(0,0,0,.75) 65%);
z-index:-3;
filter:saturate(1.2);
}}
.wrap {{
max-width: 1350px;
margin: 3% auto 4%;
padding: 0 28px;
}}
.hero {{
background: rgba(5,5,10,.45);
border: 1px solid rgba(0,255,255,.2);
border-radius: 20px;
backdrop-filter: blur(10px);
box-shadow: 0 0 40px rgba(0,255,255,.2), 0 0 60px rgba(255,0,255,.2);
padding: 32px;
}}
.topline {{
font-size: 12px;
letter-spacing: 3px;
text-transform: uppercase;
color: rgba(255,255,255,.9);
display:flex;
gap:12px;
align-items:center;
}}
.beat {{
width:9px; height:9px; border-radius:50%;
background: var(--neon1);
box-shadow: 0 0 10px var(--neon1);
animation: pulse 1s ease-in-out infinite;
}}
@keyframes pulse {{
0%,100%{{transform:scale(1); opacity:1}}
50%{{transform:scale(1.6); opacity:.4}}
}}
.title {{
font-family:'Orbitron', sans-serif;
font-size: 2.8rem;
text-transform: uppercase;
letter-spacing: 1px;
margin: 10px 0 3px;
text-shadow: 0 0 25px var(--neon1), 0 0 25px var(--neon2);
}}
.subtitle {{
font-size: 15px;
opacity: .9;
margin-bottom: 16px;
}}
.row {{
display:grid;
grid-template-columns: 320px 1fr;
gap: 24px;
align-items: center;
}}
.deck {{
display:grid;
grid-template-columns: repeat(auto-fit, minmax(230px, 1fr));
gap: 14px;
margin-top: 18px;
}}
.card {{
background: rgba(1,1,1,.25);
border: 1px solid rgba(0,255,255,.12);
border-radius: 14px;
padding: 16px;
box-shadow: 0 18px 35px rgba(0,0,0,.3);
transition:.25s ease;
}}
.card:hover {{
transform: translateY(-4px);
border-color: rgba(255,0,255,.35);
box-shadow: 0 0 25px rgba(255,0,255,.3);
}}
.card h3 {{
margin: 0 0 8px;
font-size: 14px;
letter-spacing: .5px;
color: var(--neon1);
}}
.list {{
padding-left: 18px;
font-size: 13px;
line-height: 1.4;
}}
.badge {{
display:inline-block;
background: linear-gradient(120deg, var(--neon1), var(--neon2));
color: #05121d;
font-size:10px;
font-weight:700;
text-transform:uppercase;
padding:3px 10px 2px;
border-radius:999px;
margin-bottom:6px;
}}
.vinyl {{
width: 260px;
height: 260px;
border-radius: 50%;
background:
    radial-gradient(circle at 50% 50%, #040715 0 22%, transparent 23%),
    repeating-radial-gradient(circle, #0d122a 0 2px, #02040c 2px 4px);
position: relative;
margin:auto;
animation: spin 14s linear infinite;
box-shadow: 0 0 25px rgba(0,255,255,.3);
}}
@keyframes spin {{
to {{ transform: rotate(360deg); }}
}}
.vinyl-cover {{
position:absolute; inset:38px;
border-radius: 50%;
background: url('https://images.pexels.com/photos/167404/pexels-photo-167404.jpeg?auto=compress&cs=tinysrgb&w=800') center/cover no-repeat;
box-shadow: inset 0 0 15px rgba(0,0,0,.8);
}}
.vinyl-label {{
position:absolute; inset:98px;
width:66px; height:66px;
background: conic-gradient(from 90deg, var(--neon1), var(--neon2), var(--neon1));
border-radius:50%;
display:flex; align-items:center; justify-content:center;
font-size:10px;
font-weight:700;
color:#020712;
text-align:center;
line-height:1.1;
}}
.vinyl-hole {{
position:absolute; inset:124px;
width:14px; height:14px;
border-radius:50%;
background:#020712;
box-shadow: inset 0 0 5px rgba(255,255,255,.4);
}}
.marquee-wrap {{
margin-top: 26px;
overflow:hidden;
border-radius: 14px;
border:1px solid rgba(0,255,255,.08);
}}
.marquee {{
display:flex; gap:12px;
padding:10px;
animation: slide 55s linear infinite;
width:max-content;
}}
@keyframes slide {{
from {{ transform:translateX(0); }}
to {{ transform:translateX(-50%); }}
}}
.cover {{
width:110px; height:110px;
border-radius: 12px;
background: #040613 center/cover no-repeat;
box-shadow: 0 0 18px rgba(0,0,0,.4);
outline:1px solid rgba(255,255,255,.06);
}}
.audio-box {{
margin-top:12px;
background: rgba(0,0,0,.3);
border:1px solid rgba(0,255,255,.3);
border-radius:12px;
padding:10px;
box-shadow:0 0 15px rgba(255,0,255,.3);
}}
small {{ font-size:11px; opacity:.7; }}
</style>

<div class="bg"></div>

<div class="wrap">
<div class="hero">
    <div class="topline">
    <div class="beat"></div> Proyecto Final · Inteligencia Artificial · TECHNOSELLER
    </div>
    <div class="title">TECHNOSELLER</div>
    <div class="subtitle">
    <b>Alumno:</b> Juan Cruz Cordoneda · <b>Tutor:</b> Julio Paredes
    </div>
    <div class="subtitle">
    <b>Proyecto:</b> Predicción de Éxito Musical Electrónico Basado en Ganancia Económica
    </div>
    <div class="row">
<div>
    <div class="vinyl">
    <div class="vinyl-cover"></div>
    <div class="vinyl-label">TECHNO<br>ML</div>
    <div class="vinyl-hole"></div>
    </div>
    <div class="audio-box">
    <small>✨ Presioná play y dejá que la inspiración suene</small>
    <audio controls style="width:100%; margin-top:4px;">
        <source src="https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3" type="audio/mpeg">
        Tu navegador no soporta audio.
    </audio>
    </div>
</div>

<div>
    <div class="deck">

                <div class="card">
                    <span class="badge">🎯 Objetivo</span>
                    <h3>Clasificación de Éxito Musical</h3>
                    <p>
                        Desarrollar modelos de <b>Machine Learning</b> capaces de predecir
                        si una canción electrónica será exitosa utilizando variables acústicas
                        como <b>tempo, energía, valencia y duración</b>.
                    </p>
                </div>

                <div class="card">
                    <span class="badge">🔥 Variable Objetivo</span>
                    <h3>Éxito (Popularidad ≥ 60)</h3>
                    <p>
                        Se define como <b>éxito</b> a toda canción cuya popularidad supera
                        los <b>60 puntos</b>, generando una variable binaria:
                    </p>
                    <p style="font-size:12px;">
                        • 1 = canción exitosa<br>
                        • 0 = canción no exitosa
                    </p>
                </div>

                <div class="card">
                    <span class="badge">📊 Métricas</span>
                    <h3>Evaluación Estadística</h3>
                    <p>
                        Los modelos se evaluaron utilizando métricas clásicas de clasificación:
                        <b>accuracy, precision, recall y F1-score</b>, priorizando la capacidad
                        de generalización.
                    </p>
                </div>

                <div class="card">
                    <span class="badge">⚖️ Desbalance de Clases</span>
                    <h3>Problema Real</h3>
                    <p>
                        El dataset presenta un <b>desbalance natural</b> entre canciones exitosas
                        y no exitosas, lo cual impacta directamente en el recall de la clase minoritaria.
                    </p>
                </div>

                <div class="card">
                    <span class="badge">🤖 Modelos</span>
                    <h3>Comparados</h3>
                    <ul class="list">
                        <li><b>Random Forest:</b> modelo robusto y estable.</li>
                        <li><b>XGBoost:</b> mayor precisión total y mejor capacidad predictiva.</li>
                    </ul>
                </div>

                <div class="card">
                    <span class="badge">✅ Conclusión</span>
                    <h3>Resultado Final</h3>
                    <p>
                        XGBoost logró la <b>mayor precisión total (~75%)</b>, mientras que
                        Random Forest mostró un comportamiento consistente.
                    </p>
                </div>

            </div>
</div>
"""

HTML(html)

In [2]:
# === PASO 1: CARGA Y LIMPIEZA DEL DATASET ===

# Cargar dataset proporcionado por un cliente del sector musical
# (por razones de confidencialidad, no se puede revelar el nombre ni la fuente exacta)
df = pd.read_csv("dataset.csv")

# Renombramos las columnas a nombres en español para que sea más fácil leerlas
df.columns = [
    "Cancion", "Artista", "Album", "Año", "Duracion_ms", "Compas",
    "Bailabilidad", "Energia", "Tonalidad", "Volumen", "Modo", "Habla",
    "Acustica", "Instrumentalidad", "Presencia_en_vivo", "Valencia",
    "Tempo", "Popularidad"
]

# Creamos variables nuevas combinando atributos importantes:
df["Energia_Valencia"] = df["Energia"] * df["Valencia"]          # mide energía + emoción positiva
df["Dance_Volumen"] = df["Bailabilidad"] * abs(df["Volumen"])    # mide baile + intensidad del volumen
df["Tempo_Normalizado"] = df["Tempo"] / df["Duracion_ms"]        # tempo ajustado a la duración
df["Acusticidad_Invertida"] = 1 - df["Acustica"]                 # cuánto más electrónico es el tema

# Eliminamos columnas que no ayudan al modelo (texto, redundantes o poco útiles)
df = df.drop([
    "Cancion",
    "Artista",
    "Album",
    "Instrumentalidad",   # ya está representada por Acústica
    "Bailabilidad",        # ya la usamos en Dance_Volumen
    "Presencia_en_vivo"    # aporta poco al género electrónico
], axis=1)

# Variable objetivo que marca si la canción fue exitosa (1) o no (0)
df["Exito"] = (df["Popularidad"] >= 60).astype(int)

df.head()


,Año,Duracion_ms,Compas,Energia,Tonalidad,Volumen,Modo,Habla,Acustica,Valencia,Tempo,Popularidad,Energia_Valencia,Dance_Volumen,Tempo_Normalizado,Acusticidad_Invertida,Exito
0,2009,412266,4,0.816,0,-5.749,1,0.0476,0.004710,0.5190,128.012,9,0.423504,3.736850,0.000311,0.995290,0
1,2009,206400,4,0.822,3,-8.630,0,0.0497,0.255000,0.3700,137.909,33,0.304140,4.366780,0.000668,0.745000,0
2,2014,400226,4,0.581,5,-6.563,1,0.0386,0.000701,0.2650,130.014,16,0.153965,4.535033,0.000325,0.999299,0
3,2013,192866,4,0.932,7,-5.797,0,0.0658,0.065700,0.3350,132.991,38,0.312220,3.715877,0.000690,0.934300,0
4,2019,332946,4,0.525,1,-10.987,1,0.0408,0.089400,0.0867,125.985,34,0.045518,5.295734,0.000378,0.910600,0


In [3]:
# === PASO 2: ANÁLISIS EXPLORATORIO DE DATOS ===

# Crear diccionario de datos
diccionario_datos = {
    "Columna": [
        "Año", "Duracion_ms", "Compas", "Energia", "Tonalidad",
        "Volumen", "Modo", "Habla", "Acustica", "Valencia",
        "Tempo", "Popularidad", "Energia_Valencia",
        "Dance_Volumen", "Tempo_Normalizado",
        "Acusticidad_Invertida", "Exito"
    ],
    "Descripción": [
        "Año de lanzamiento de la canción",
        "Duración total del tema en milisegundos",
        "Compás o métrica musical del tema",
        "Nivel general de energía (0 = baja, 1 = alta)",
        "Tonalidad predominante de la canción",
        "Volumen promedio del tema en dB",
        "Modo musical: 1 = mayor, 0 = menor",
        "Proporción de voz hablada en la pista",
        "Nivel de componentes acústicos (0 = electrónico, 1 = acústico)",
        "Valencia emocional: felicidad o positividad",
        "Velocidad rítmica en BPM",
        "Popularidad del tema (0–100)",
        "Interacción entre energía y valencia",
        "Interacción entre bailabilidad y volumen",
        "Tempo relativo a la duración",
        "Inversión de acusticidad",
        "Variable objetivo: 1 = exitosa, 0 = no exitosa"
    ]
}


diccionario_df = pd.DataFrame(diccionario_datos)

display(
    diccionario_df.style
    .set_caption("📘 Diccionario de Datos del Dataset")
    .set_table_styles([
        {"selector": "caption", "props": [("color", "#00ffff"), ("font-size", "16px"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "th", "props": [("background-color", "#05050a"), ("color", "#00ffff"), ("font-weight", "bold"), ("text-align", "left")]},
        {"selector": "td", "props": [("background-color", "#0a0a0f"), ("color", "#e0f7ff"), ("border", "1px solid #111"), ("padding", "8px"), ("text-align", "left")]},
        {"selector": "table", "props": [("width", "100%"), ("border-collapse", "collapse"), ("margin", "0 auto")]}
    ])
    .hide(axis="index")
    .set_properties(subset=["Columna"], **{"width": "30%", "font-weight": "bold", "color": "#00ffff", "text-align": "left"})
    .set_properties(subset=["Descripción"], **{"width": "100%", "text-align": "left"})
)

# === Visualización del dataset ===
print("\n----------------------------------------------------------------------------------------------------------------------------------------")
print("**Gráfico 1: Distribución de Popularidad según Éxito**")
print("Este gráfico muestra cómo se distribuyen los niveles de popularidad y cuántas canciones superan el umbral de éxito (Popularidad ≥ 60).")
print("Permite observar que la mayoría de los temas se concentran en valores medios, mientras los éxitos se destacan en la zona alta.")
print("------------------------------------------------------------------------------------------------------------------------------------------")

fig = px.histogram(
    df, x="Popularidad", color="Exito", nbins=20,
    title="Distribución de Popularidad según Éxito",
    color_discrete_map={0: "#00bfff", 1: "#ff0080"}
)
fig.update_layout(
    title_font=dict(size=18, color="white"),
    paper_bgcolor="#000000",
    plot_bgcolor="#000000",
    font=dict(color="#e0f7ff"),
    xaxis=dict(showgrid=False, color="#00ffff"),
    yaxis=dict(showgrid=True, gridcolor="#222", color="#00ffff"),
    margin=dict(l=40, r=40, t=70, b=40)
)
fig.show()

# Distribución de clases
print("Distribución de la variable objetivo (Éxito):")
display(df["Exito"].value_counts(normalize=True).rename("Proporción"))

# --- Mapa de Correlaciones ---
print("\n------------------------------------------------------------------------------------------------------------------------------------------")
print("**Gráfico 2: Mapa de Correlaciones**")
print("Este mapa revela la relación entre variables numéricas del dataset. Los tonos claros indican correlaciones positivas (por ejemplo, energía y valencia),")
print("mientras que los oscuros representan correlaciones negativas. Ayuda a identificar predictores relevantes y redundancias entre variables.")
print("------------------------------------------------------------------------------------------------------------------------------------------")

corr = df.drop("Exito", axis=1).corr().round(3)
fig = px.imshow(
    corr, text_auto=True, color_continuous_scale="Tealrose",
    title="Mapa de Correlaciones entre Variables Numéricas",
    aspect="auto",
    width=950, height=950
)
fig.update_traces(textfont=dict(size=11))
fig.update_layout(
    title_font=dict(size=18, color="white"),
    paper_bgcolor="#000000",
    plot_bgcolor="#000000",
    font=dict(color="#e0f7ff"),
    xaxis=dict(color="#e0f7ff", tickangle=45),
    yaxis=dict(color="#e0f7ff"),
    margin=dict(l=40, r=40, t=70, b=40)
)
fig.show()


Columna,Descripción
Año,Año de lanzamiento de la canción
Duracion_ms,Duración total del tema en milisegundos
Compas,Compás o métrica musical del tema
Energia,"Nivel general de energía (0 = baja, 1 = alta)"
Tonalidad,Tonalidad predominante de la canción
Volumen,Volumen promedio del tema en dB
Modo,"Modo musical: 1 = mayor, 0 = menor"
Habla,Proporción de voz hablada en la pista
Acustica,"Nivel de componentes acústicos (0 = electrónico, 1 = acústico)"
Valencia,Valencia emocional: felicidad o positividad



----------------------------------------------------------------------------------------------------------------------------------------
**Gráfico 1: Distribución de Popularidad según Éxito**
Este gráfico muestra cómo se distribuyen los niveles de popularidad y cuántas canciones superan el umbral de éxito (Popularidad ≥ 60).
Permite observar que la mayoría de los temas se concentran en valores medios, mientras los éxitos se destacan en la zona alta.
------------------------------------------------------------------------------------------------------------------------------------------


Distribución de la variable objetivo (Éxito):


Exito
0    0.748571
1    0.251429
Name: Proporción, dtype: float64


------------------------------------------------------------------------------------------------------------------------------------------
**Gráfico 2: Mapa de Correlaciones**
Este mapa revela la relación entre variables numéricas del dataset. Los tonos claros indican correlaciones positivas (por ejemplo, energía y valencia),
mientras que los oscuros representan correlaciones negativas. Ayuda a identificar predictores relevantes y redundancias entre variables.
------------------------------------------------------------------------------------------------------------------------------------------


In [4]:
# === PASO 3: DIVISIÓN TRAIN Y TEST ===

# Separar variables
X = df.drop(["Exito", "Popularidad"], axis=1)
y = df["Exito"]

# División entrenamiento / test (estratificada)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Escalado (fit SOLO en train)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# === PASO 4: MODELO BASE - RANDOM FOREST (EVALUACIÓN ESTADÍSTICA) ===

from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

param_grid = {
    "n_estimators": [300, 500, 800],
    "max_depth": [8, 10, 12],
    "min_samples_split": [2, 4],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

# Predicciones con umbral estándar
y_pred = best_model.predict(X_test)

print("Mejores hiperparámetros encontrados:")
print(grid_search.best_params_)

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))


Fitting 5 folds for each of 72 candidates, totalling 360 fits
Mejores hiperparámetros encontrados:
{'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}

Reporte de clasificación:
              precision    recall  f1-score   support

           0       0.79      0.89      0.83       157
           1       0.45      0.28      0.35        53

    accuracy                           0.73       210
   macro avg       0.62      0.58      0.59       210
weighted avg       0.70      0.73      0.71       210


Matriz de confusión:
[[139  18]
 [ 38  15]]


In [6]:
# === PASO 5: MODELO FINAL RANDOM FOREST ===

best_params_base = {
    "n_estimators": 500,
    "max_depth": 8,
    "max_features": "sqrt",
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "class_weight": "balanced",
    "criterion": "gini",
    "random_state": 42
}

rf_model = RandomForestClassifier(**best_params_base)
rf_model.fit(X_train, y_train)

# Predicciones con umbral estándar (0.5)
y_pred_final = rf_model.predict(X_test)

# Evaluación estadística
cm = confusion_matrix(y_test, y_pred_final)
print("\n MATRIZ DE CONFUSION:\n", cm)

print("\n REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred_final))

rf_accuracy = accuracy_score(y_test, y_pred_final)
print(f"\n PRECISIÓN TOTAL (ACCURACY): {rf_accuracy:.3f}")

print(f"\n SENSIBILIDAD (RECALL): {recall_score(y_test, y_pred_final):.3f}")



 MATRIZ DE CONFUSION:
 [[139  18]
 [ 38  15]]

 REPORTE DE CLASIFICACIÓN:
              precision    recall  f1-score   support

           0       0.79      0.89      0.83       157
           1       0.45      0.28      0.35        53

    accuracy                           0.73       210
   macro avg       0.62      0.58      0.59       210
weighted avg       0.70      0.73      0.71       210


 PRECISIÓN TOTAL (ACCURACY): 0.733

 SENSIBILIDAD (RECALL): 0.283


In [ ]:
# === PASO 6: BUSQUEDA DE HIPERPARÁMETROS XGBOOST (OPTIMIZADO POR GANANCIA) ===
from sklearn.metrics import make_scorer

# ─── MATRIZ DE GANANCIA ─────────────────────────────────────────────────────
# Ajustá estos valores según el negocio real:
#
#   TP  → Apostamos a un éxito y acierta   → ganancia por promoción/inversión
#   FP  → Apostamos a un éxito y fracasa   → pérdida por inversión fallida
#   TN  → Descartamos un fracaso y acertamos → sin costo
#   FN  → Descartamos un éxito real         → costo de oportunidad perdida
# ────────────────────────────────────────────────────────────────────────────

GANANCIA_TP = 10    # ganancia por éxito correctamente identificado
COSTO_FP    = -5    # pérdida por invertir en un tema que fracasa
GANANCIA_TN =  0    # correcto rechazo — sin inversión ni pérdida
COSTO_FN    = -2    # costo de oportunidad por no apostar a un éxito real

def scorer_ganancia_fn(y_true, y_pred):
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return (tp * GANANCIA_TP + fp * COSTO_FP +
            tn * GANANCIA_TN + fn * COSTO_FN)

scorer_ganancia = make_scorer(scorer_ganancia_fn)

# 1) Espacio de búsqueda de hiperparámetros
param_dist = {
    "n_estimators":      [200, 300, 400, 600, 800],
    "max_depth":         [3, 4, 5, 6],
    "learning_rate":     [0.03, 0.05, 0.1],
    "subsample":         [0.7, 0.8, 0.9],
    "colsample_bytree":  [0.6, 0.7, 0.8, 0.9],
    "gamma":             [0, 0.1, 0.2],
    "min_child_weight":  [1, 2, 3, 4]
}

# 2) Modelo base XGBoost
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
)

# 3) Búsqueda aleatoria optimizando GANANCIA TOTAL
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=40,
    scoring=scorer_ganancia,   # 🔹 optimiza ganancia económica total
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

best_xgb = random_search.best_estimator_

print("\n MEJORES PARÁMETROS ENCONTRADOS:")
print(random_search.best_params_)

# 4) Evaluación estadística estándar
y_pred = best_xgb.predict(X_test)

print("\n MATRIZ DE CONFUSIÓN:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\n REPORTE DE CLASIFICACIÓN:")
print(classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
print(f"\n PRECISIÓN TOTAL (ACCURACY): {accuracy:.3f}")

# 5) Evaluación de ganancia económica en test
tp = int(cm[1, 1])
fp = int(cm[0, 1])
tn = int(cm[0, 0])
fn = int(cm[1, 0])

ganancia_test = scorer_ganancia_fn(y_test, y_pred)

print(f"\n GANANCIA TOTAL EN TEST: {ganancia_test}")
print(f"   TP={tp:3d}  → +{tp * GANANCIA_TP:5}")
print(f"   FP={fp:3d}  →  {fp * COSTO_FP:5}")
print(f"   TN={tn:3d}  → +{tn * GANANCIA_TN:5}")
print(f"   FN={fn:3d}  →  {fn * COSTO_FN:5}")


In [8]:
# === PASO 7: MODELO FINAL XGBOOST ===

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.9,
    gamma=0.1,
    min_child_weight=4,
    random_state=42,
    eval_metric="logloss"
)

# Entrenar modelo
xgb_model.fit(X_train, y_train)

# Predicciones estándar (umbral 0.5)
y_pred_final = xgb_model.predict(X_test)

# Evaluación estadística
cm = confusion_matrix(y_test, y_pred_final)

print("\n MATRIZ DE CONFUSIÓN:\n", cm)
print("\n REPORTE DE CLASIFICACIÓN:\n")
print(classification_report(y_test, y_pred_final))

xbg_accuracy = accuracy_score(y_test, y_pred_final)
print(f"\n PRECISIÓN TOTAL (ACCURACY): {xbg_accuracy:.3f}")



 MATRIZ DE CONFUSIÓN:
 [[142  15]
 [ 37  16]]

 REPORTE DE CLASIFICACIÓN:

              precision    recall  f1-score   support

           0       0.79      0.90      0.85       157
           1       0.52      0.30      0.38        53

    accuracy                           0.75       210
   macro avg       0.65      0.60      0.61       210
weighted avg       0.72      0.75      0.73       210


 PRECISIÓN TOTAL (ACCURACY): 0.752


In [9]:
# === PASO 8: COMPARACIÓN DE PRECISIÓN (ACCURACY) ===

accuracy_models = {
    "Random Forest": rf_accuracy,
    "XGBoost": xbg_accuracy
}

colores = ["#007bff", "#ff1744"]

fig = go.Figure(data=[
    go.Bar(
        x=list(accuracy_models.keys()),
        y=list(accuracy_models.values()),
        text=[f"{v:.3f}" for v in accuracy_models.values()],
        textposition="auto",
        marker_color=colores,
        marker_line=dict(color="#000000", width=1.8)
    )
])

fig.update_layout(
    title="Comparación de Precisión (Accuracy) por Modelo",
    xaxis_title="Modelo",
    yaxis_title="Precisión (Accuracy)",
    title_font=dict(size=20, color="white", family="Orbitron"),
    paper_bgcolor="#000000",
    plot_bgcolor="#000000",
    font=dict(color="#e0f7ff", family="Poppins"),
    xaxis=dict(color="#00ffff"),
    yaxis=dict(color="#00ffff", gridcolor="#111", range=[0, 1]),
    margin=dict(l=40, r=40, t=70, b=40),
    showlegend=False
)

fig.update_traces(
    hovertemplate="<b>%{x}</b><br>Accuracy: %{y:.3f}<extra></extra>",
    marker_line_width=2,
    marker_line_color="#0d0d0d"
)

fig.show()


In [10]:
html_conclusiones = """
<div class="hero">
  <div class="topline">
    <div class="beat"></div> TECHNOSELLER · Cierre del Proyecto
  </div>
  <div class="title">Conclusiones Finales</div>
  <div class="subtitle">Resultados del modelo, variables clave y aplicaciones prácticas</div>

  <div class="deck" style="display:flex; flex-wrap:wrap; gap:16px; justify-content:space-between;">

    <div class="card" style="flex:1 1 23%; min-width:250px;">
      <span class="badge">📊 Resultados</span>
      <h3>Desempeño de los Modelos</h3>
      <p>
        Los modelos entrenados lograron un desempeño sólido en un contexto de
        <b>clases desbalanceadas</b>.  
        <b>XGBoost</b> alcanzó la <b>mayor precisión total (~75%)</b>, mientras que
        <b>Random Forest</b> mostró un comportamiento estable y consistente.
      </p>
    </div>

    <div class="card" style="flex:1 1 23%; min-width:250px;">
      <span class="badge">🎧 Variables Clave</span>
      <h3>Factores de Éxito Musical</h3>
      <p>
        Las variables con mayor influencia en la predicción fueron
        <b>energía</b>, <b>valencia</b> y <b>bailabilidad</b>.  
        Estas características describen canciones <b>enérgicas, positivas y rítmicas</b>,
        asociadas a una mayor probabilidad de éxito.
      </p>
    </div>

    <div class="card" style="flex:1 1 23%; min-width:250px;">
      <span class="badge">⚖️ Aprendizajes</span>
      <h3>Más Allá del Accuracy</h3>
      <p>
        El proyecto evidenció que, en datasets reales, una buena
        <b>precisión global</b> no siempre implica un alto <b>recall</b> en la clase minoritaria.  
        El desbalance de clases sigue siendo el principal desafío a abordar.
      </p>
    </div>

    <div class="card" style="flex:1 1 23%; min-width:250px;">
      <span class="badge">🚀 Impacto</span>
      <h3>Aplicación Práctica</h3>
      <p>
        <b>TECHNOSELLER</b> demuestra cómo los modelos de <b>Machine Learning</b> pueden
        asistir en la evaluación objetiva de canciones, aportando soporte
        analítico para decisiones estratégicas en la industria musical.
      </p>
    </div>

  </div>
</div>
"""
HTML(html_conclusiones)


In [11]:

# Si tu mejor modelo es XGBoost
joblib.dump(xgb_model, "modelo_random_forest.pkl")
joblib.dump(rf_model, "modelo_xgboost.pkl")
joblib.dump(scaler, "scaler.pkl")

print("✅ Modelos guardados correctamente en formato .pkl")


✅ Modelos guardados correctamente en formato .pkl
